In [2]:

import pandas as pd
import numpy as np
import plotly.express as px

# Load dataset
df_supply = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1', on_bad_lines='skip')

# Print actual column names to see exact spelling
print("Actual columns in dataset:")
print([col for col in df_supply.columns if 'Shipping' in col or 'Status' in col or 'Days' in col])

# Preview using the first few available columns safely
df_supply.head(3)

Actual columns in dataset:
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Delivery Status', 'Order Status', 'Product Status', 'Shipping Mode']


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [4]:


# 1. Inspect the unique values of delivery status to see how orders are categorized
print("Original Delivery Status counts:")
print(df_supply['Delivery Status'].value_counts())

# 2. Create a binary target column for late deliveries (1 = Late, 0 = On Time / Advance)
# We check if the status indicates a late delivery
df_supply['is_late'] = df_supply['Delivery Status'].apply(lambda x: 1 if 'Late' in str(x) else 0)

# 3. Calculate delay gap (Real shipping days minus Scheduled shipping days)
# Corrected column name to 'Days for shipment (scheduled)'
df_supply['delay_gap_days'] = df_supply['Days for shipping (real)'] - df_supply['Days for shipment (scheduled)']

# 4. Calculate overall On-Time Delivery Percentage
total_orders = len(df_supply)
late_orders = df_supply['is_late'].sum()
on_time_orders = total_orders - late_orders
on_time_percentage = (on_time_orders / total_orders) * 100

print("-" * 40)
print(f"Total Orders Analyzed: {total_orders:,}")
print(f"Late Orders: {late_orders:,}")
print(f"On-Time / Early Orders: {on_time_orders:,}")
print(f"Overall On-Time Delivery Rate: {on_time_percentage:.2f}%")
print("-" * 40)

Original Delivery Status counts:
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64
----------------------------------------
Total Orders Analyzed: 180,519
Late Orders: 98,977
On-Time / Early Orders: 81,542
Overall On-Time Delivery Rate: 45.17%
----------------------------------------


In [5]:

import plotly.express as px

# 1. Group by Shipping Mode and calculate total orders vs. late orders
shipping_analysis = df_supply.groupby('Shipping Mode').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

# 2. Calculate late delivery percentage per shipping mode
shipping_analysis['late_percentage'] = (shipping_analysis['late_orders'] / shipping_analysis['total_orders']) * 100
shipping_analysis = shipping_analysis.sort_values(by='late_percentage', ascending=False)

# 3. Create a bar chart showing delay rates by shipping method
fig_mode = px.bar(
    shipping_analysis,
    x='Shipping Mode',
    y='late_percentage',
    title='<b>Late Delivery Rate by Shipping Mode</b>',
    labels={'Shipping Mode': 'Shipping Mode', 'late_percentage': 'Late Delivery Rate (%)'},
    text_auto='.1f',
    color='late_percentage',
    color_continuous_scale='Reds'
)

fig_mode.update_layout(template='plotly_white')
fig_mode.show()

In [6]:
# Project 3 - Milestone 3: Segment Delays by Department
import plotly.express as px

# 1. Group by Department Name and calculate total orders vs. late orders
dept_analysis = df_supply.groupby('Department Name').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

# 2. Calculate late delivery percentage per department
dept_analysis['late_percentage'] = (dept_analysis['late_orders'] / dept_analysis['total_orders']) * 100
dept_analysis = dept_analysis.sort_values(by='late_percentage', ascending=False)

# 3. Create a bar chart showing delay rates by department
fig_dept = px.bar(
    dept_analysis,
    x='Department Name',
    y='late_percentage',
    title='<b>Late Delivery Rate by Department</b>',
    labels={'Department Name': 'Department', 'late_percentage': 'Late Delivery Rate (%)'},
    text_auto='.1f',
    color='late_percentage',
    color_continuous_scale='Blues'
)

fig_dept.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_dept.show()

In [7]:
# Project 3 - Milestone 4 & 6: Operations Hotspot Summary Table
# Group by Shipping Mode and Department to find the exact high-delay combination hotspots

hotspot_table = df_supply.groupby(['Shipping Mode', 'Department Name']).agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

hotspot_table['late_percentage'] = (hotspot_table['late_orders'] / hotspot_table['total_orders']) * 100

# Sort to find the highest risk combinations
hotspot_table = hotspot_table.sort_values(by='late_percentage', ascending=False)

print("Top 5 Highest Delay Operational Hotspots:")
print(hotspot_table.head(5).to_string(index=False))

Top 5 Highest Delay Operational Hotspots:
Shipping Mode Department Name  total_orders  late_orders  late_percentage
  First Class       Book Shop            53           53       100.000000
  First Class         Fitness           397          384        96.725441
  First Class        Pet Shop            84           81        96.428571
  First Class        Outdoors          1458         1398        95.884774
  First Class        Footwear          2252         2150        95.470693


In [8]:
# Project 3 - Step 5: Decision Tree Classifier to Predict Delay Risk
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Select key columns for prediction and drop missing values
model_df = df_supply[['Shipping Mode', 'Department Name', 'is_late']].dropna().copy()

# 2. Encode text categories into numbers so machine learning algorithms can read them
le_mode = LabelEncoder()
le_dept = LabelEncoder()

model_df['Shipping_Mode_Enc'] = le_mode.fit_transform(model_df['Shipping Mode'])
model_df['Dept_Enc'] = le_dept.fit_transform(model_df['Department Name'])

# Define features (X) and target variable to predict (y)
X = model_df[['Shipping_Mode_Enc', 'Dept_Enc']]
y = model_df['is_late']

# 3. Split data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and train the Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

# 5. Test and evaluate the model's predictive power
y_pred = dt_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Decision Tree Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Decision Tree Model Accuracy: 69.16%

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.88      0.72     16307
           1       0.84      0.54      0.66     19797

    accuracy                           0.69     36104
   macro avg       0.73      0.71      0.69     36104
weighted avg       0.74      0.69      0.69     36104



In [18]:
# Standalone Graph: Shipping Mode
import plotly.express as px

df_ship = df_supply.groupby('Shipping Mode').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

df_ship['late_percentage'] = (df_ship['late_orders'] / df_ship['total_orders']) * 100

fig_ship = px.bar(
    df_ship, x='Shipping Mode', y='late_percentage',
    title='<b>Late Delivery Rate by Shipping Mode</b>',
    text_auto='.1f', color='late_percentage', color_continuous_scale='Reds'
)
fig_ship.update_layout(template='plotly_white')
fig_ship.show()

In [19]:
# Standalone Graph: Department
import plotly.express as px

df_dept = df_supply.groupby('Department Name').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

df_dept['late_percentage'] = (df_dept['late_orders'] / df_dept['total_orders']) * 100

fig_dept = px.bar(
    df_dept, x='Department Name', y='late_percentage',
    title='<b>Late Delivery Rate by Department</b>',
    text_auto='.1f', color='late_percentage', color_continuous_scale='Blues'
)
fig_dept.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_dept.show()

In [20]:
# Standalone Graph: Day of the Week
import pandas as pd
import plotly.express as px

df_supply['parsed_date'] = pd.to_datetime(df_supply['order date (DateOrders)'], errors='coerce')
df_supply['day_of_week'] = df_supply['parsed_date'].dt.day_name()

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_dow = df_supply.groupby('day_of_week').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reindex(dow_order).reset_index()

df_dow['late_percentage'] = (df_dow['late_orders'] / df_dow['total_orders']) * 100

fig_dow = px.bar(
    df_dow, x='day_of_week', y='late_percentage',
    title='<b>Late Delivery Rate by Day of Week</b>',
    text_auto='.1f', color='late_percentage', color_continuous_scale='Viridis'
)
fig_dow.update_layout(template='plotly_white')
fig_dow.show()

In [21]:
# Standalone Graph: Season
import pandas as pd
import plotly.express as px

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

if 'parsed_date' not in df_supply.columns:
    df_supply['parsed_date'] = pd.to_datetime(df_supply['order date (DateOrders)'], errors='coerce')

df_supply['season'] = df_supply['parsed_date'].dt.month.apply(get_season)

df_season = df_supply.groupby('season').agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

df_season['late_percentage'] = (df_season['late_orders'] / df_season['total_orders']) * 100

fig_season = px.bar(
    df_season, x='season', y='late_percentage',
    title='<b>Late Delivery Rate by Season</b>',
    text_auto='.1f', color='late_percentage', color_continuous_scale='Oranges'
)
fig_season.update_layout(template='plotly_white')
fig_season.show()

In [22]:
# Standalone Graph: Warehouse Block
import plotly.express as px

if 'Warehouse block' in df_supply.columns:
    df_wh = df_supply.groupby('Warehouse block').agg(
        total_orders=('Order Id', 'count'),
        late_orders=('is_late', 'sum')
    ).reset_index()

    df_wh['late_percentage'] = (df_wh['late_orders'] / df_wh['total_orders']) * 100

    fig_wh = px.bar(
        df_wh, x='Warehouse block', y='late_percentage',
        title='<b>Late Delivery Rate by Warehouse Block</b>',
        text_auto='.1f', color='late_percentage', color_continuous_scale='Teals'
    )
    fig_wh.update_layout(template='plotly_white')
    fig_wh.show()
else:
    print("Warehouse block segment graph is not possible (Column not found in dataset).")

Warehouse block segment graph is not possible (Column not found in dataset).


In [23]:
# Standalone Graph: Market / Region
import plotly.express as px

market_col = 'Market' if 'Market' in df_supply.columns else 'Customer Country'

df_market = df_supply.groupby(market_col).agg(
    total_orders=('Order Id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()

df_market['late_percentage'] = (df_market['late_orders'] / df_market['total_orders']) * 100
df_market = df_market.sort_values(by='late_percentage', ascending=False)

fig_market = px.bar(
    df_market, x=market_col, y='late_percentage',
    title=f'<b>Late Delivery Rate by {market_col}</b>',
    text_auto='.1f', color='late_percentage', color_continuous_scale='Purples'
)
fig_market.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_market.show()

In [24]:
# Project 3: Executive Mini-Dashboard for Operations Analytics
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Compute Core KPIs
total_orders = len(df_supply)
late_orders = int(df_supply['is_late'].sum())
on_time_percentage = ((total_orders - late_orders) / total_orders) * 100
avg_delay_gap = df_supply['delay_gap_days'].mean()

# Print Executive Summary Banner
print("=" * 60)
print("       EXECUTIVE OPERATIONS DASHBOARD - SUMMARY KPIS        ")
print("=" * 60)
print(f"📦 Total Orders Analyzed     : {total_orders:,}")
print(f"🚨 Total Late Deliveries     : {late_orders:,} ({100 - on_time_percentage:.2f}%)")
print(f"✅ On-Time Delivery Rate     : {on_time_percentage:.2f}%")
print(f"⏱️ Average Delay Gap         : {avg_delay_gap:.2f} days")
print("=" * 60)

# 2. Create a Side-by-Side Subplot Dashboard (Shipping Mode vs Department)
fig_dash = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Late Rate by Shipping Mode (%)</b>', '<b>Late Rate by Department (%)</b>')
)

# Shipping Mode Data Preparation
df_ship = df_supply.groupby('Shipping Mode').agg(total=('Order Id', 'count'), late=('is_late', 'sum')).reset_index()
df_ship['late_pct'] = (df_ship['late'] / df_ship['total']) * 100

# Department Data Preparation
df_dept = df_supply.groupby('Department Name').agg(total=('Order Id', 'count'), late=('is_late', 'sum')).reset_index()
df_dept['late_pct'] = (df_dept['late'] / df_dept['total']) * 100

# Add Shipping Mode Bar Chart
fig_dash.add_trace(
    go.Bar(x=df_ship['Shipping Mode'], y=df_ship['late_pct'], marker_color='#d62728', text=df_ship['late_pct'].round(1), textposition='auto'),
    row=1, col=1
)

# Add Department Bar Chart
fig_dash.add_trace(
    go.Bar(x=df_dept['Department Name'], y=df_dept['late_pct'], marker_color='#1f77b4', text=df_dept['late_pct'].round(1), textposition='auto'),
    row=1, col=2
)

# Update layout for professional executive presentation
fig_dash.update_layout(
    title_text='<b>Supply Chain Operations & Bottleneck Dashboard</b>',
    showlegend=False,
    template='plotly_white',
    height=500
)

fig_dash.update_xaxes(tickangle=-45)
fig_dash.show()

       EXECUTIVE OPERATIONS DASHBOARD - SUMMARY KPIS        
📦 Total Orders Analyzed     : 180,519
🚨 Total Late Deliveries     : 98,977 (54.83%)
✅ On-Time Delivery Rate     : 45.17%
⏱️ Average Delay Gap         : 0.57 days
